# Capstone: Custom gadgets and hardware noise

This final unit brings together the **Develop → Test → Deploy** workflow from the previous units in a real-world QEC research scenario: extending an existing protocol with a custom hardware-motivated instruction, verifying its fault tolerance, and benchmarking it under realistic device noise.

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do we extend a protocol with a hardware-motivated logical instruction and evaluate it end to end?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Define a custom `Clifford` instruction (`hadamard_swap_all`) and add its transversal gadget to `C4`.
- Derive stabilizer checks with `ec.filled` and verify protocol completeness and distance two.
- Benchmark a Q# logical circuit comparing uniform depolarizing noise against asymmetric neutral-atom gate noise.

## Setting up

The cell below loads a fresh copy of the upgraded distance-two `C4` protocol.

In [ ]:
import qodec
from qdk import TargetProfile, ec, qsharp
from qdk.simulation import NoiseConfig, run_qir
from _unit import make_decoder, make_upgraded_protocol

capstone_protocol = make_upgraded_protocol()
layer0 = capstone_protocol.layers[0]

## Designing a zone-global Hadamard-swap gadget

You are a QEC researcher and you want to extend the `C4` protocol with a new logical instruction—`hadamard_swap_all`—that exploits a signature hardware capability of a neutral-atom quantum processor: illuminating an entire four-atom code zone with a single global optical pulse (`H 0 1 2 3`) that has an order-of-magnitude lower error rate than two-qubit Rydberg entangling gates.

Earlier, in *Building a Baseline Qodec and Instruction Sets*, `bare-css/v1` omitted `h_all` because `H 0 1 2 3` swaps row and column Paulis on `C4`:

<div style="text-align:center;"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 940 288" width="100%" style="max-width:100%;height:auto" class="qdk-ec-hswap" role="img" data-asset="tutorial_hswap_geometry.svg" aria-label="Schematic of the zone-global H 0 1 2 3 pulse acting in parallel on qubits q0 through q3 and its induced two-qubit logical Clifford and stabilizer-swap contract on C4."> <style> .qdk-ec-hswap { --qdk-fg: var(--vscode-editor-foreground, var(--jp-content-font-color1, currentColor)); --qdk-muted: color-mix(in srgb, var(--qdk-fg) 80%, transparent); --qdk-border: color-mix(in srgb, var(--qdk-fg) 30%, transparent); font-family: "Segoe UI", -apple-system, BlinkMacSystemFont, Roboto, Helvetica, Arial, sans-serif; } .qdk-ec-hswap .cluster-blue { fill: color-mix(in srgb, #1976d2 8%, transparent); stroke: color-mix(in srgb, #1976d2 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-hswap .cluster-teal { fill: color-mix(in srgb, #0d9488 8%, transparent); stroke: color-mix(in srgb, #0d9488 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-hswap .cluster-purple { fill: color-mix(in srgb, #7c3aed 8%, transparent); stroke: color-mix(in srgb, #7c3aed 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-hswap .cluster-neutral { fill: color-mix(in srgb, var(--qdk-fg) 4%, transparent); stroke: color-mix(in srgb, var(--qdk-fg) 32%, transparent); stroke-width: 1.8; } .qdk-ec-hswap .cluster-red { fill: color-mix(in srgb, #dc2626 9%, transparent); stroke: color-mix(in srgb, #dc2626 62%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-hswap .stage-blue { fill: color-mix(in srgb, #1976d2 18%, transparent); stroke: color-mix(in srgb, #1976d2 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-hswap .stage-teal { fill: color-mix(in srgb, #0d9488 18%, transparent); stroke: color-mix(in srgb, #0d9488 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-hswap .stage-purple { fill: color-mix(in srgb, #7c3aed 18%, transparent); stroke: color-mix(in srgb, #7c3aed 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-hswap .stage-red { fill: color-mix(in srgb, #dc2626 16%, transparent); stroke: color-mix(in srgb, #dc2626 78%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-hswap .loop-x { fill: color-mix(in srgb, #1976d2 20%, transparent); stroke: color-mix(in srgb, #1976d2 85%, var(--qdk-fg)); stroke-width: 2.2; } .qdk-ec-hswap .loop-z { fill: color-mix(in srgb, #0d9488 20%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2.2; stroke-dasharray: 6 3; } .qdk-ec-hswap .hdr-blue { fill: color-mix(in srgb, #1976d2 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-hswap .hdr-teal { fill: color-mix(in srgb, #0d9488 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-hswap .hdr-purple { fill: color-mix(in srgb, #7c3aed 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-hswap .hdr-neutral { fill: var(--qdk-fg); font-size: 12.2px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-hswap .hdr-red { fill: color-mix(in srgb, #dc2626 75%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-hswap .box-title { fill: var(--qdk-fg); font-size: 11.6px; font-weight: 700; } .qdk-ec-hswap .box-sub { fill: var(--qdk-muted); font-size: 10px; } .qdk-ec-hswap .code-tag { fill: var(--qdk-fg); font-family: Consolas, "Courier New", monospace; font-size: 9.4px; font-weight: 600; } .qdk-ec-hswap .badge-blue { fill: #1976d2; } .qdk-ec-hswap .badge-teal { fill: #0d9488; } .qdk-ec-hswap .badge-purple { fill: #7c3aed; } .qdk-ec-hswap .badge-red { fill: #dc2626; } .qdk-ec-hswap .badge-txt { fill: #ffffff; font-size: 9.6px; font-weight: 700; } .qdk-ec-hswap .conn { stroke: color-mix(in srgb, var(--qdk-fg) 68%, transparent); stroke-width: 2.2; fill: none; } .qdk-ec-hswap .wire { stroke: color-mix(in srgb, var(--qdk-fg) 48%, transparent); stroke-width: 1.5; fill: none; } .qdk-ec-hswap .gate-neutral { stroke: color-mix(in srgb, var(--qdk-fg) 78%, transparent); stroke-width: 1.8; fill: none; } .qdk-ec-hswap .gate-red { stroke: #dc2626; stroke-width: 2.3; fill: none; } .qdk-ec-hswap .gate-teal { stroke: #0d9488; stroke-width: 2.4; fill: none; } .qdk-ec-hswap .qubit-node { fill: var(--vscode-editor-background, #1e1e1e); stroke: color-mix(in srgb, var(--qdk-fg) 75%, transparent); stroke-width: 2; } .qdk-ec-hswap .anc-node { fill: color-mix(in srgb, #0d9488 22%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2; } </style> <defs> <marker id="hsw-arr" viewBox="0 0 10 10" refX="7.5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto-start-reverse"> <path d="M 0 1.5 L 8.5 5 L 0 8.5 z" fill="var(--qdk-fg)"/> </marker> </defs> <rect x="14" y="14" width="356" height="260" rx="10" class="cluster-purple"/> <text x="192" y="36" text-anchor="middle" class="hdr-purple">PHYSICAL GADGET &#8212; ZONE-GLOBAL PULSE</text> <rect x="130" y="48" width="128" height="148" rx="8" class="stage-purple" stroke-dasharray="5 3"/> <text x="194" y="66" text-anchor="middle" class="code-tag">H 0 1 2 3</text> <text x="38" y="100" class="code-tag">q0</text> <line x1="66" y1="96" x2="338" y2="96" class="wire"/> <rect x="178" y="86" width="32" height="20" rx="4" class="qubit-node"/> <text x="194" y="100" text-anchor="middle" class="box-title">H</text> <text x="38" y="126" class="code-tag">q1</text> <line x1="66" y1="122" x2="338" y2="122" class="wire"/> <rect x="178" y="112" width="32" height="20" rx="4" class="qubit-node"/> <text x="194" y="126" text-anchor="middle" class="box-title">H</text> <text x="38" y="152" class="code-tag">q2</text> <line x1="66" y1="148" x2="338" y2="148" class="wire"/> <rect x="178" y="138" width="32" height="20" rx="4" class="qubit-node"/> <text x="194" y="152" text-anchor="middle" class="box-title">H</text> <text x="38" y="178" class="code-tag">q3</text> <line x1="66" y1="174" x2="338" y2="174" class="wire"/> <rect x="178" y="164" width="32" height="20" rx="4" class="qubit-node"/> <text x="194" y="178" text-anchor="middle" class="box-title">H</text> <rect x="28" y="208" width="328" height="52" rx="7" class="stage-purple"/> <text x="192" y="228" text-anchor="middle" class="box-title">Conjugates X_q &#8596; Z_q on all 4 qubits</text> <text x="192" y="248" text-anchor="middle" class="code-tag">Zero 2Q gates &#183; gadget distance = 2</text> <path d="M 374 144 L 408 144" class="conn" marker-end="url(#hsw-arr)"/> <rect x="412" y="14" width="514" height="260" rx="10" class="cluster-neutral"/> <text x="669" y="36" text-anchor="middle" class="hdr-neutral">INDUCED QODEC CONTRACT ON C4</text> <rect x="428" y="48" width="482" height="134" rx="8" class="stage-blue"/> <text x="669" y="68" text-anchor="middle" class="box-title">Logical Clifford Action (Hadamard + Qubit 0 &#8596; 1 Swap)</text> <text x="446" y="92" class="code-tag">x[0]: X_0 X_1 &#8594; Z_0 Z_1 = z[1]</text> <text x="680" y="92" class="code-tag">x[1]: X_0 X_2 &#8594; Z_0 Z_2 = z[0]</text> <text x="446" y="112" class="code-tag">z[0]: Z_0 Z_2 &#8594; X_0 X_2 = x[1]</text> <text x="680" y="112" class="code-tag">z[1]: Z_0 Z_1 &#8594; X_0 X_1 = x[0]</text> <line x1="444" y1="124" x2="894" y2="124" class="wire"/> <text x="669" y="143" text-anchor="middle" class="box-sub">Declared logical-index Clifford action:</text> <text x="669" y="164" text-anchor="middle" class="code-tag">Clifford({&quot;X_0&quot;: &quot;Z_1&quot;, &quot;Z_0&quot;: &quot;X_1&quot;, &quot;X_1&quot;: &quot;Z_0&quot;, &quot;Z_1&quot;: &quot;X_0&quot;})</text> <rect x="428" y="192" width="482" height="68" rx="8" class="stage-teal"/> <text x="669" y="210" text-anchor="middle" class="box-title">Stabilizer Swap Derived by ec.filled (s0: XXXX &#8596; s1: ZZZZ)</text> <text x="669" y="230" text-anchor="middle" class="code-tag">check 0: [out[0].stabilizers[0], in[0].stabilizers[1]]</text> <text x="669" y="248" text-anchor="middle" class="code-tag">check 1: [out[0].stabilizers[1], in[0].stabilizers[0]]</text> </svg></div>

<div style="text-align:center;">

*A single zone-global `H 0 1 2 3` pulse applies parallel Hadamards across `q0..q3`, swapping the two stabilizers and mapping logical generators `x[0], z[0] <-> z[1], x[1]` (declared as `X_0, Z_0 <-> Z_1, X_1` in `qodec.actions.Clifford`).*

</div>

By declaring that exact `Clifford` action on `hadamard_swap_all`, we can add the zero-CNOT `H 0 1 2 3` gadget to Layer 0, derive its stabilizer checks with `ec.filled`, and audit the seven-gadget protocol:

In [ ]:
hswap_instr = qodec.Instruction(
    mnemonic="hadamard_swap_all",
    inputs=[qodec.instructions.BlockOperand("C4")],
    outputs=[qodec.instructions.BlockOperand("C4")],
    action=[qodec.actions.Clifford({"X_0": "Z_1", "Z_0": "X_1", "X_1": "Z_0", "Z_1": "X_0"})],
)
layer0.instruction_set.instructions["hadamard_swap_all"] = hswap_instr

hswap_gadget = qodec.Gadget(
    implements=hswap_instr,
    inputs=list(layer0.gadgets["syndrome"].inputs),
    outputs=list(layer0.gadgets["syndrome"].outputs),
    circuit=qodec.gadgets.Circuit(
        format="stim",
        source="H 0 1 2 3\n",
        instruction_set=capstone_protocol.layers[1].instruction_set,
    ),
)
layer0.gadgets["hadamard_swap_all"] = ec.filled(hswap_gadget)

report = ec.audit(capstone_protocol)
print("Capstone protocol audit:", report)
print("Derived checks:", list(layer0.gadgets["hadamard_swap_all"].checks))

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why did ec.filled pair input stabilizer 0 with output stabilizer 1 on hadamard_swap_all?</summary>

<div style="padding:0.1em 1em;">

Because `H 0 1 2 3` conjugates the input $X$ stabilizer (`stabilizers[0] = X_0 X_1 X_2 X_3`) into the output $Z$ stabilizer (`stabilizers[1] = Z_0 Z_1 Z_2 Z_3`) and vice versa.

</div>

</details></div>

## Benchmarking under asymmetric hardware noise

Now let's compile a Q# experiment (`TestHadamardSwapSeq`) that prepares $\vert 00\rangle_L$, applies an odd sequence of three `hadamard_swap_all` pulses to map $\vert 00\rangle_L \to \vert ++\rangle_L$, extracts syndromes, and measures in the $X$ basis.

Unlike a uniform depolarizing model where every primitive fails at rate $p = 10^{-2}$, neutral-atom hardware typically exhibits strong rate asymmetry—for example, single-qubit gates at $p_{1\mathrm{Q}} = 5\times 10^{-4}$, two-qubit entangling gates at $p_{2\mathrm{Q}} = 10^{-2}$, and reset noise at $p_{\mathrm{reset}} = 3\times 10^{-3}$ (note that `NoiseConfig` models `H 0 1 2 3` as four independent single-qubit `h` channels at $p_{1\mathrm{Q}}$ rather than correlated global-beam over-rotation). Let's compare both noise profiles across `2,000` shots:

In [ ]:
qsharp.init(target_profile=TargetProfile.Adaptive_RI)
qsharp.eval("""
@Measurement() operation prepare_z_all(block : Qubit) : Result { body intrinsic; }
@Measurement() operation measure_x_all(block : Qubit) : (Result, Result) { body intrinsic; }
operation syndrome(block : Qubit) : Unit { body intrinsic; }
operation hadamard_swap_all(block : Qubit) : Unit { body intrinsic; }

operation TestHadamardSwapSeq() : (Result[], Result[]) {
    use block = Qubit();
    let reject = prepare_z_all(block);
    hadamard_swap_all(block);
    hadamard_swap_all(block);
    hadamard_swap_all(block);
    syndrome(block);
    let (x0, x1) = measure_x_all(block);
    ([reject], [x0, x1])
}
""")
hswap_qir = str(qsharp.compile("TestHadamardSwapSeq()"))
scored_decoder = make_decoder(max_readout_score=0.05)

def evaluate_noise(label: str, p_1q: float, p_2q: float, p_reset: float) -> None:
    noise = NoiseConfig()
    for g in ("h", "x", "y", "z"):
        getattr(noise, g).set_depolarizing(p_1q)
    noise.cx.set_depolarizing(p_2q)
    noise.mresetz.x = p_reset
    rows = run_qir(
        hswap_qir,
        qodec=capstone_protocol,
        decoder=scored_decoder,
        noise=noise,
        type="stabilizer",
        shots=2000,
        seed=123,
        on_shot_failure="discard",
    )
    accepted = [
        logical for flags, logical in rows if all(b == qsharp.Result.Zero for b in flags)
    ]
    wrong = sum(any(b != qsharp.Result.Zero for b in logical) for logical in accepted)
    ler = wrong / len(accepted)
    print(f"{label:<28} -> accepted={len(accepted)}/2000, LER={ler:.5f}")

evaluate_noise("Uniform depolarizing (p=1e-2)", 1e-2, 1e-2, 1e-2)
evaluate_noise("Asymmetric neutral-atom noise", 5e-4, 1e-2, 3e-3)

Because `hadamard_swap_all` uses zero two-qubit gates, lowering $p_{1\mathrm{Q}}$ ($10^{-2} \to 5\times 10^{-4}$) and $p_{\mathrm{reset}}$ ($10^{-2} \to 3\times 10^{-3}$)—even with $p_{2\mathrm{Q}}$ held fixed at $10^{-2}$—simultaneously boosts shot acceptance and cuts the conditional logical error rate substantially.

This capstone completes the protocol loop from *Tutorial Overview*: when a logical instruction's formal action, gadget circuit, parity equations, and gadget distance all agree inside a `qodec`, you can compile and benchmark new hardware-tailored operations end to end without rewriting your simulation pipeline.

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why did switching to the asymmetric neutral-atom noise profile improve both shot acceptance and logical error rate even though two-qubit CX noise stayed at 0.01?</summary>

<div style="padding:0.1em 1em;">

Because `hadamard_swap_all` (`H 0 1 2 3`) uses zero two-qubit gates, lowering $p_{1\mathrm{Q}}$ ($10^{-2} \to 5\times 10^{-4}$) and $p_{\mathrm{reset}}$ ($10^{-2} \to 3\times 10^{-3}$) directly suppresses faults across all three Hadamard-swap pulses as well as single-qubit `H` and reset operations in preparation, syndrome extraction, and $X$-basis readout—cutting both single-fault syndromes (fewer rejected shots) and two-fault logical errors.

</div>

</details></div>

## Exercise: Verify gadget distance of the Hadamard-swap gadget

Complete `hadamard_swap_gadget_distance` by deriving the parity checks for `hswap_gadget` with `ec.filled` and returning its integer gadget distance.

In [ ]:
from qdk import ec
from _unit import exercise, make_unfilled_hswap_gadget


@exercise
def hadamard_swap_gadget_distance() -> int:
    hswap_gadget = make_unfilled_hswap_gadget()
    # Fill hswap_gadget with ec.filled and return its integer gadget distance (.value)
    return ...

**Hint**

Call `filled_g = ec.filled(hswap_gadget)` and return `ec.GadgetProfile(filled_g).distance().value`.

In [ ]:
from qdk import ec
from _unit import exercise, make_unfilled_hswap_gadget


@exercise
def hadamard_swap_gadget_distance() -> int:
    hswap_gadget = make_unfilled_hswap_gadget()
    filled_g = ec.filled(hswap_gadget)
    return ec.GadgetProfile(filled_g).distance().value

`2`. Because `H 0 1 2 3` acts independently on each physical qubit of the `C4` block with no two-qubit gates, a single fault affects at most one qubit and is detected by the stabilizer checks.

## Further reading

- [Authoring and editing qodec objects in Python ↗](https://github.com/microsoft/qdk-ec/blob/main/qodec/bindings/python/docs/usage.rst)
- [Configure simulator noise for qodec execution ↗](https://learn.microsoft.com/en-us/azure/quantum/run-decode-logical-programs#configure-simulator-noise)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../09-simulation-and-decoding/simulation_and_decoding.workbook.ipynb" title="Previous unit: Simulating and Decoding Under Noise" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Simulating and Decoding Under Noise</span></span></a><span></span></div>